# Demo 10d — Gaussian mixtures (EM) and fuzzy c-means

**Class 10 · D1** (slides: `slides/10_clustering.md`)

* EM by hand: the E-step gives the responsibilities $u_{ik}$, the M-step re-estimates weights, means, covariances;
  the log-likelihood never falls. Checked against `GaussianMixture`.
* The four covariance structures on the anisotropic set: spherical, diagonal, tied, full.
* Fuzzy $c$-means and its fuzzifier $m$: from hard $k$-means ($m \to 1$) to flat memberships ($m$ large).
* Failure cases: bad starts, singular components, non-Gaussian clusters.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import polars as pl
from scipy.optimize import nnls
from scipy.stats import multivariate_normal as mvn
from sklearn.cluster import KMeans
from sklearn.datasets import make_blobs
from sklearn.metrics import adjusted_rand_score as ari
from sklearn.mixture import GaussianMixture

from faa import clustering as cl
from faa import demo, plotlib
from faa.demo import show, slide

demo.setup("10")
BLUE, ORANGE, GREEN, RED, SKY, PURPLE, GREY = (
    plotlib.BLUE,
    plotlib.ORANGE,
    plotlib.GREEN,
    plotlib.RED,
    plotlib.SKY,
    plotlib.PURPLE,
    plotlib.GREY,
)
PAL = np.array([[0.0, 0.447, 0.698], [0.902, 0.624, 0.0], [0.0, 0.62, 0.451]])
SEED = 0
B = cl.benchmarks(SEED)

## Slide: EM by hand

$E$-step: $u_{ik} = \pi_k\,\mathcal{N}(x_i \mid \mu_k, C_k) / \sum_j \pi_j\,\mathcal{N}(x_i \mid \mu_j, \Sigma_j)$.
$M$-step: $N_k = \sum_i u_{ik}$, $\pi_k = N_k/n$, $\mu_k = \sum_i u_{ik} x_i / N_k$,
$C_k = \sum_i u_{ik}(x_i - \mu_k)(x_i - \mu_k)^\top / N_k$.

In [ ]:
def em_gmm(X, mu, iters=100, tol=1e-8, reg=1e-6):
    """Full-covariance EM from given means; returns pi, mu, Sigma, responsibilities and the log-likelihood per iteration."""
    n, d = X.shape
    k = len(mu)
    pi = np.full(k, 1 / k)
    Sigma = np.array([np.cov(X.T) + reg * np.eye(d) for _ in range(k)])
    ll = []
    for _ in range(iters):
        dens = np.column_stack([pi[j] * mvn.pdf(X, mu[j], Sigma[j]) for j in range(k)])
        ll.append(float(np.log(dens.sum(1)).sum()))
        U = dens / dens.sum(1, keepdims=True)
        Nk = U.sum(0)
        pi = Nk / n
        mu = (U.T @ X) / Nk[:, None]
        Sigma = np.array([((U[:, [j]] * (X - mu[j])).T @ (X - mu[j])) / Nk[j] + reg * np.eye(d) for j in range(k)])
        if len(ll) > 1 and ll[-1] - ll[-2] < tol:
            break
    return pi, mu, Sigma, U, ll


X, y = B["blobs"]
start = KMeans(3, n_init=1, random_state=SEED).fit(X).cluster_centers_
pi, mu, Sigma, U, ll = em_gmm(X, start)
gm = GaussianMixture(
    3, covariance_type="full", means_init=start, random_state=SEED, tol=1e-8, max_iter=200, reg_covar=1e-6
).fit(X)
slide("EM by hand")
print(
    f"iterations {len(ll)}, log-likelihood {ll[0]:.1f} -> {ll[-1]:.1f}; never falls: {all(b >= a - 1e-9 for a, b in zip(ll, ll[1:]))}"
)
print(
    f"sklearn: lower bound per sample x n = {gm.score(X) * len(X):.1f}; max |u - u_sklearn| = {np.abs(U - gm.predict_proba(X)).max():.2e}"
)
print(f"weights pi {np.round(pi, 3)}; sklearn {np.round(gm.weights_, 3)}")
f, ax = plt.subplots(1, 2, figsize=(plotlib.FULL, 1.5), gridspec_kw={"width_ratios": [1, 1.1]})
ax[0].plot(ll, "o-", color=BLUE, ms=3)
ax[0].set_xlabel("EM iteration"), ax[0].set_ylabel("log-likelihood")
ax[1].scatter(*X.T, c=np.clip(U @ PAL, 0, 1), s=6, lw=0)
for m_, S_ in zip(mu, Sigma):
    w, v = np.linalg.eigh(S_)
    t = np.linspace(0, 2 * np.pi, 80)
    ax[1].plot(*(m_[:, None] + v @ (2 * np.sqrt(w)[:, None] * np.array([np.cos(t), np.sin(t)]))), color="k", lw=0.8)
ax[1].set_xticks([]), ax[1].set_yticks([])
plotlib.save(f, "d1_em")

## Slide: One E-step on five points

Five points in one dimension, two components with $\pi = (0.5, 0.5)$, $\mu = (0, 4)$, $\sigma = 1$. The
responsibility of the first component is a logistic function of the distance to the boundary.

In [ ]:
x1 = np.array([-0.5, 0.8, 2.0, 3.0, 4.5])
dens = np.column_stack([0.5 * np.exp(-((x1 - m_) ** 2) / 2) / np.sqrt(2 * np.pi) for m_ in (0.0, 4.0)])
u1 = dens / dens.sum(1, keepdims=True)
slide("One E-step on five points")
show(
    pl.DataFrame(
        {"x": x1, "p1 = pi N(x|0,1)": dens[:, 0], "p2 = pi N(x|4,1)": dens[:, 1], "u_1": u1[:, 0], "u_2": u1[:, 1]}
    ),
    digits=4,
)

## Slide: Four covariance structures

`spherical`: one variance per component ($k$-means with weights); `diag`: axis-aligned ellipses; `tied`: one
shared ellipse; `full`: any ellipse. Parameters grow from $k(d+1)$ to $k\,d(d+1)/2$; the anisotropic set needs `full`.

In [ ]:
Xa, ya = B["aniso"]
rows = []
f, ax = plt.subplots(1, 4, figsize=(plotlib.FULL, 1.3), sharex=True, sharey=True)
for a, cov in zip(ax, ("spherical", "diag", "tied", "full")):
    g = GaussianMixture(3, covariance_type=cov, n_init=5, random_state=SEED).fit(Xa)
    Ua = g.predict_proba(Xa)
    order = [
        int(np.argmin(((g.means_ - c) ** 2).sum(1)))
        for c in KMeans(3, n_init=5, random_state=SEED).fit(Xa).cluster_centers_
    ]
    rows.append(
        (
            cov,
            3 * 2 + {"spherical": 3, "diag": 6, "tied": 3, "full": 9}[cov] + 2,
            ari(ya, Ua.argmax(1)),
            g.bic(Xa),
            cl.entropy(Ua).mean(),
        )
    )
    a.scatter(*Xa.T, c=np.clip(Ua[:, order] @ PAL, 0, 1), s=4, lw=0)
    a.set_title(cov), a.set_xticks([]), a.set_yticks([])
plotlib.save(f, "d1_covariance")
slide("Four covariance structures")
show(
    pl.DataFrame(rows, schema=["covariance", "free parameters", "ARI", "BIC", "mean entropy (bits)"], orient="row"),
    digits=3,
)

## Slide: Fuzzy $c$-means and the fuzzifier $m$

FCM minimises $J_m = \sum_i\sum_k u_{ik}^m\lVert x_i - c_k\rVert^2$ with $\sum_k u_{ik} = 1$. The updates are
$u_{ik} = 1/\sum_j (d_{ik}/d_{ij})^{2/(m-1)}$ and $c_k = \sum_i u_{ik}^m x_i / \sum_i u_{ik}^m$.
$m \to 1$ is $k$-means; $m \to \infty$ gives $u_{ik} \to 1/k$.

In [ ]:
Xb, yb = B["varied"]
rows = []
for m_ in (1.05, 1.5, 2.0, 3.0, 5.0, 10.0):
    c_, Uf, cost = cl.fcm(Xb, 3, m=m_, iters=300, seed=SEED)
    rows.append((m_, ari(yb, Uf.argmax(1)), Uf.max(1).mean(), float((Uf**2).sum() / len(Xb)), len(cost)))
slide("Fuzzy c-means and the fuzzifier m")
show(pl.DataFrame(rows, schema=["m", "ARI", "mean top u", "partition coefficient", "iterations"], orient="row"))
print(
    f"k-means ARI {ari(yb, KMeans(3, n_init=10, random_state=SEED).fit_predict(Xb)):.3f}; GMM ARI {ari(yb, GaussianMixture(3, n_init=3, random_state=SEED).fit_predict(Xb)):.3f}"
)
f, ax = plt.subplots(1, 4, figsize=(plotlib.FULL, 1.3), sharex=True, sharey=True)
for a, m_ in zip(ax, (1.05, 1.5, 2.0, 5.0)):
    _, Uf, _ = cl.fcm(Xb, 3, m=m_, iters=300, seed=SEED)
    a.scatter(*Xb.T, c=np.clip(Uf @ PAL, 0, 1), s=4, lw=0)
    a.set_title(f"$m$ = {m_}"), a.set_xticks([]), a.set_yticks([])
plotlib.save(f, "d1_fuzzifier")

## Slide: FCM, GMM, NMF: three ways to write $u_{ik}$

Same overlapping blobs, three soft clusterings. All three rows sum to 1; they differ in what they assume: FCM
the distance, GMM the Gaussian density, NMF the non-negative parts.

In [ ]:
import pynnmf

Xo, yo = make_blobs(300, centers=3, cluster_std=1.2, random_state=SEED)
_, Ufc, _ = cl.fcm(Xo, 3, m=2.0, seed=SEED)
Ugm = GaussianMixture(3, n_init=3, random_state=SEED).fit(Xo).predict_proba(Xo)
Xc = Xo - Xo.mean(0)
_, Wn, Hn, _ = pynnmf.nmf_hals(np.c_[np.maximum(Xc, 0), np.maximum(-Xc, 0)].T, 3, n=300, seed=SEED, init="random")
Unm = cl.nmf_membership(Wn, Hn)


def best_corr(A, Bm):
    return np.round(
        [
            np.corrcoef(A[:, j], Bm[:, np.argmax([np.corrcoef(A[:, j], Bm[:, q])[0, 1] for q in range(3)])])[0, 1]
            for j in range(3)
        ],
        3,
    )


slide("FCM, GMM, NMF")
show(
    pl.DataFrame(
        [
            ("FCM (m = 2)", ari(yo, Ufc.argmax(1)), Ufc.max(1).mean(), cl.entropy(Ufc).mean()),
            ("GMM full", ari(yo, Ugm.argmax(1)), Ugm.max(1).mean(), cl.entropy(Ugm).mean()),
            ("NMF, normalised H", ari(yo, Unm.argmax(1)), Unm.max(1).mean(), cl.entropy(Unm).mean()),
        ],
        schema=["method", "ARI", "mean top u", "mean entropy (bits)"],
        orient="row",
    )
)
print("membership correlation, FCM vs GMM per cluster:", best_corr(Ufc, Ugm), " NMF vs GMM:", best_corr(Unm, Ugm))

## Failure 1: the start decides the optimum

EM and FCM find a local optimum. One random start per run (`init_params="random"`, `n_init=1`), 100 seeds, on
the `varied` set: the spread of the final log-likelihood and the share of bad runs.

In [ ]:
Xv, yv = B["varied"]
res = []
for s in range(100):
    g = GaussianMixture(3, init_params="random_from_data", n_init=1, random_state=s).fit(Xv)
    res.append((g.score(Xv), ari(yv, g.predict(Xv))))
res = np.array(res)
slide("The start decides the optimum")
print(
    f"mean log-likelihood per point: best {res[:, 0].max():.3f}, median {np.median(res[:, 0]):.3f}, worst {res[:, 0].min():.3f}"
)
print(
    f"runs more than 0.01 below the best: {(res[:, 0] < res[:, 0].max() - 0.01).mean():.2f}; ARI range {res[:, 1].min():.3f} .. {res[:, 1].max():.3f}"
)

## Failure 2: a component collapses on one point; clusters that are not Gaussian

A component with one point has covariance 0: the likelihood grows without bound. `reg_covar` adds a floor.
And a mixture can fit any density with enough components, but a component is not a cluster: on the moons two
Gaussians cut across, ten Gaussians cover the shape but give ten clusters.

In [ ]:
x0 = 1.7  # a data point; a component with mean x0 and standard deviation sigma gives it the density 1 / (sigma sqrt(2 pi))
slide("A component that collapses on one point")
show(
    pl.DataFrame(
        {
            "sigma": ["1", "0.1", "0.01", "0.001", "1e-6"],
            "density of the point": [1 / (s_ * np.sqrt(2 * np.pi)) for s_ in (1.0, 0.1, 0.01, 1e-3, 1e-6)],
        }
    ),
    digits=1,
)
print("the likelihood grows without bound as sigma -> 0; GaussianMixture adds reg_covar = 1e-6 to every variance")
Xm, ym = B["moons"]
rows = []
for k in (2, 5, 10):
    g = GaussianMixture(k, n_init=3, random_state=SEED).fit(Xm)
    rows.append((k, ari(ym, g.predict(Xm)), g.bic(Xm)))
slide("Gaussian components are not clusters")
show(pl.DataFrame(rows, schema=["components", "ARI to the two moons", "BIC"], orient="row"))

## Failure 3: the fuzzifier in high dimension

Eleven standardised wine measurements, $k = 2$, a sample of 3 000 wines, red against white as the check. $k$-means
finds the colours; fuzzy $c$-means does too for $m$ near 1, but from $m \approx 1.5$ the centres slide together and
the partition no longer follows the colour.

In [ ]:
from sklearn.preprocessing import StandardScaler

from faa import data

Xw, _, _ = data.load_table("wine_quality", "quality")
Xw = StandardScaler().fit_transform(Xw)
red = np.r_[np.ones(1599), np.zeros(len(Xw) - 1599)].astype(int)
S = np.random.default_rng(SEED).choice(len(Xw), 3000, replace=False)
Xs, rs = Xw[S], red[S]
kmw = KMeans(2, n_init=10, random_state=SEED).fit(Xs)
rows = [("k-means", np.nan, ari(rs, kmw.labels_), np.nan, float(np.linalg.norm(np.diff(kmw.cluster_centers_, axis=0))))]
for m_ in (1.1, 1.3, 1.5, 2.0):
    c_, Uw, _ = cl.fcm(Xs, 2, m=m_, iters=500, seed=SEED)
    rows.append(
        ("fuzzy c-means", m_, ari(rs, Uw.argmax(1)), float(Uw.max(1).mean()), float(np.linalg.norm(c_[0] - c_[1])))
    )
slide("The fuzzifier in high dimension")
show(
    pl.DataFrame(
        rows, schema=["method", "m", "ARI to red/white", "mean top u", "distance between centres"], orient="row"
    )
)

## Slide: One E-step on five points (figure)

The two weighted densities and the membership $u_1$ along the line; the five points of the table sit on it.

In [ ]:
xs = np.linspace(-3, 7, 300)
p1 = 0.5 * np.exp(-(xs**2) / 2) / np.sqrt(2 * np.pi)
p2 = 0.5 * np.exp(-((xs - 4) ** 2) / 2) / np.sqrt(2 * np.pi)
f, ax = plt.subplots(figsize=(plotlib.FULL, 1.4))
(
    ax.plot(xs, p1, color=BLUE, label="$\\pi_1 \\mathcal{N}(x|0,1)$"),
    ax.plot(xs, p2, color=ORANGE, label="$\\pi_2 \\mathcal{N}(x|4,1)$"),
)
ax2 = ax.twinx()
ax2.plot(xs, p1 / (p1 + p2), color="k", ls="--", lw=0.9, label="$u_1$")
ax2.scatter(x1, u1[:, 0], color=RED, zorder=3, s=14)
ax2.set_ylabel("$u_1$"), ax2.grid(False), ax2.spines["right"].set_visible(True)
ax.set_xlabel("$x$"), ax.legend(loc="upper center", ncol=2)
plotlib.save(f, "d1_estep")

## Slide: Where the membership update comes from (figure)

For two clusters, $u_1 = 1/(1 + (d_1/d_2)^{2/(m-1)})$: the share of the closer centre as the distance ratio grows.

In [ ]:
ratio = np.linspace(0.2, 5, 200)
f, ax = plt.subplots(figsize=(1.35, 1.15))
for m_, c in ((1.1, RED), (2.0, BLUE), (5.0, GREEN)):
    ax.plot(ratio, 1 / (1 + ratio ** (2 / (m_ - 1))), color=c, label=f"$m$ = {m_}")
ax.set_xlabel("$d_1 / d_2$"), ax.set_ylabel("$u_1$"), ax.legend()
plotlib.save(f, "d1_fcm_u")
slide("Where the membership update comes from")
print("u_1 at d_1/d_2 = 0.5:", {m_: round(float(1 / (1 + 0.5 ** (2 / (m_ - 1)))), 3) for m_ in (1.1, 2.0, 5.0)})

## Slide: Three ways to write $u_{ik}$ (figure)

The memberships of FCM, GMM and NMF on the line between two centroids of the overlapping blobs.

In [ ]:
cen = KMeans(3, n_init=10, random_state=SEED).fit(Xo).cluster_centers_
line = cen[0] + np.linspace(-0.2, 1.2, 80)[:, None] * (cen[1] - cen[0])
gmo = GaussianMixture(3, n_init=3, random_state=SEED).fit(Xo)
d2 = ((line[:, None] - cen[None]) ** 2).sum(-1)
u_f = (d2**-1.0) / (d2**-1.0).sum(1, keepdims=True)
cols = [int(np.argmin(((gmo.means_ - c) ** 2).sum(1))) for c in cen]
u_g = gmo.predict_proba(line)[:, cols]

Xl = line - Xo.mean(0)
lifted = np.c_[np.maximum(Xl, 0), np.maximum(-Xl, 0)]
u_n = np.array([cl.normalise_rows((nnls(Wn, row)[0] * Wn.sum(0))[None])[0] for row in lifted])
lab0 = KMeans(3, n_init=10, random_state=SEED).fit(Xo).labels_
j0 = int(np.argmax([Unm[lab0 == 0, j].mean() for j in range(3)]))
f, ax = plt.subplots(figsize=(plotlib.COL, 1.5))
t = np.linspace(-0.2, 1.2, 80)
(
    ax.plot(t, u_f[:, 0], color=BLUE, label="FCM"),
    ax.plot(t, u_g[:, 0], color=ORANGE, label="GMM"),
    ax.plot(t, u_n[:, j0], color=GREEN, label="NMF"),
)
ax.set_xlabel("position between centroids 0 and 1"), ax.set_ylabel("membership of cluster 0"), ax.legend()
plotlib.save(f, "d1_three_u")
slide("Three ways: membership of cluster 0 along the line")
mid = np.argmin(abs(t - 0.5))
print(f"at the midpoint: FCM {u_f[mid, 0]:.3f}, GMM {u_g[mid, 0]:.3f}, NMF {u_n[mid, j0]:.3f}")

## Figures: components against clusters, and the fuzzifier

ARI and BIC against the number of components on the moons; mean top membership against $m$.

In [ ]:
ncomp = [1, 2, 3, 5, 8, 10]
aris, bics = [], []
for k_ in ncomp:
    g_ = GaussianMixture(k_, n_init=3, random_state=SEED).fit(Xm)
    aris.append(ari(ym, g_.predict(Xm))), bics.append(g_.bic(Xm))
f, ax = plt.subplots(figsize=(plotlib.COL, 1.5))
ax.plot(ncomp, aris, "o-", color=BLUE, label="ARI")
ax2 = ax.twinx()
ax2.plot(ncomp, bics, "s--", color=ORANGE, label="BIC")
ax2.grid(False), ax2.spines["right"].set_visible(True), ax2.set_ylabel("BIC")
ax.set_xlabel("components"), ax.set_ylabel("ARI")
plotlib.save(f, "d1_components")
ms = [1.05, 1.5, 2.0, 3.0, 5.0, 10.0]
top = [cl.fcm(Xb, 3, m=m_, iters=300, seed=SEED)[1].max(1).mean() for m_ in ms]
ari_w = [ari(rs, cl.fcm(Xs, 2, m=m_, iters=500, seed=SEED)[1].argmax(1)) for m_ in ms]
f, ax = plt.subplots(figsize=(plotlib.COL, 1.5))
ax.plot(ms, top, "o-", color=BLUE, label="top $u$ (varied)")
ax.plot(ms, ari_w, "s-", color=RED, label="ARI (wine)")
ax.set_xscale("log"), ax.set_xlabel("fuzzifier $m$"), ax.legend()
plotlib.save(f, "d1_mcurve")
slide("The fuzzifier curve")
print("wine ARI by m:", dict(zip(ms, np.round(ari_w, 3))))